# RAGAS 평가 코드 (업로드용)

## 개요
생성된 테스트셋(질문·정답·근거 세트)을 RAGAS 지표로 평가하는 독립 실행용 노트북입니다. Faithfulness / AnswerRelevancy / ContextPrecision / ContextRecall 네 가지 지표를 계산하고, 지표별 Acceptable Threshold 기준으로 합격/불합격을 판정한 뒤 결과를 CSV로 저장합니다. 다중쿼리·단일쿼리 노트북에 포함된 평가 셀과 동일한 로직을 독립 파일로 분리해둔 버전입니다.

## API 키 설정
코드에서 사용하는 `api_key` 변수(OpenAI API 키)는 보안을 위해 실제 키 값을 코드에서 제거했습니다. 실행하려면 아래처럼 본인의 환경변수에서 값을 불러오는 코드를 직접 추가해야 합니다.
```python
import os
api_key = os.environ["OPENAI_API_KEY"]
```
이 코드는 `api_key`를 처음 사용하는 셀(LLM·임베딩 초기화) 이전에 실행되어야 합니다.

## 주요 기능
1. **라이브러리 설치** — ragas, langchain 계열 패키지를 설치합니다.
2. **RAGAS 평가 실행** — 테스트셋 JSON을 로드해 4가지 지표로 평가하고, Acceptable Threshold 기준 합격 여부를 판정한 뒤 CSV로 결과를 저장·다운로드합니다.

## 설계 포인트
- **Threshold 기반 판정**: 단순 평균 점수만 보지 않고, 메트릭별 Acceptable Threshold를 두어 항목별 합격/불합격을 판정합니다.
- **입력 스키마 주의**: 이 노트북은 `user_input` / `reference` / `reference_contexts`(ragas 원본 필드명)로 된 테스트셋 JSON을 기대합니다. 다중쿼리·단일쿼리 노트북이 저장하는 `질문`/`정답`/`근거`(한글 키) 형식과는 다르므로, 두 파이프라인을 연결할 때는 필드명 변환이 필요합니다.


**1. 필요 라이브러리 설치**


In [2]:
!pip install ragas
!pip install langchain
!pip install langchain_community
!pip install langchain-openai
!pip install jq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0

**2. 평가세트(RAGAS) 평가 실행**


(결과는 CSV로 별도 다운로드됨)


In [7]:
import json
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from datasets import Dataset
import pandas as pd
from google.colab import files

# -------------------
# Acceptable Threshold 설정
# -------------------
ACCEPTABLE_THRESHOLDS = {
    "faithfulness": 0.6,
    "answer_relevancy": 0.4,
    "context_precision": 0.8,
    "context_recall": 0.8
}

# -------------------
# LLM 및 Embeddings 초기화
# -------------------
critic_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", openai_api_key=api_key)
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=api_key
)

# -------------------
# Metrics 설정
# -------------------
metrics = [
    Faithfulness(llm=critic_llm),
    AnswerRelevancy(llm=critic_llm, embeddings=embeddings),
    ContextPrecision(llm=critic_llm),
    ContextRecall(llm=critic_llm)
]

print("✅ Metrics 초기화 완료!")

# -------------------
# JSON 문서 로드
# -------------------
with open("/content/복지제도_top3.json", "r", encoding="utf-8") as f:
    testset_korean = json.load(f)

print(f"✅ Testset 로드 완료: {len(testset_korean)}개")

eval_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

# -------------------
# 데이터 변환 (🔥 핵심 수정 부분)
# -------------------
for item in testset_korean:
    question = item.get("user_input", "")
    answer = item.get("reference", "")
    contexts = item.get("reference_contexts", [])

    if isinstance(contexts, str):
        contexts = [contexts]

    # 안전 필터
    if not question or not answer or not contexts:
        continue

    eval_data["question"].append(question)
    eval_data["answer"].append(answer)
    eval_data["contexts"].append(contexts)
    eval_data["ground_truth"].append(answer)

print(f"📊 최종 평가 샘플 수: {len(eval_data['question'])}")

if len(eval_data["question"]) == 0:
    raise ValueError("❌ 평가할 데이터가 없습니다. JSON 키 이름을 확인하세요.")

eval_dataset = Dataset.from_dict(eval_data)

# -------------------
# 평가 실행 (🔥 중복 인자 제거)
# -------------------
results = evaluate(
    dataset=eval_dataset,
    metrics=metrics
)

results_df = results.to_pandas()

# -------------------
# 평균 점수 출력 (기준선 적용)
# -------------------
print("\n📈 메트릭별 평균 점수 (Acceptable 기준 적용)\n")

for metric, threshold in ACCEPTABLE_THRESHOLDS.items():
    if metric in results_df.columns:
        score = results_df[metric].mean()
        status = "ACCEPTABLE ✅" if score >= threshold else "NOT ACCEPTABLE ❌"
        print(f"{metric}: {score:.4f} → {status} (기준 {threshold})")

# -------------------
# 개별 결과 출력 (처음 5개)
# -------------------
print("\n" + "=" * 80)
print("📋 개별 평가 결과 (처음 5개)")
print("=" * 80)

for idx in range(min(5, len(results_df))):
    print(f"\n[항목 {idx + 1}]")
    print(f"질문: {eval_data['question'][idx][:80]}...")
    print(f"답변: {eval_data['answer'][idx][:80]}...")

    row = results_df.iloc[idx]

    for metric, threshold in ACCEPTABLE_THRESHOLDS.items():
        value = row.get(metric)
        if pd.notna(value):
            status = "OK" if value >= threshold else "LOW"
            print(f"  {metric}: {value:.3f} ({status})")

# -------------------
# 낮은 점수 항목 분석
# -------------------
print("\n" + "=" * 80)
print("⚠️ 기준 미달 항목")
print("=" * 80)

for metric, threshold in ACCEPTABLE_THRESHOLDS.items():
    if metric in results_df.columns:
        low = results_df[
            results_df[metric].notna() &
            (results_df[metric] < threshold)
        ]
        print(f"\n📌 {metric} < {threshold}: {len(low)}개")

# -------------------
# 결과 저장
# -------------------
combined_results = []

for i in range(len(results_df)):
    row = results_df.iloc[i]
    result = {
        "질문": eval_data["question"][i],
        "답변": eval_data["answer"][i],
        "근거_개수": len(eval_data["contexts"][i])
    }

    for metric, threshold in ACCEPTABLE_THRESHOLDS.items():
        value = row.get(metric)
        result[metric] = value
        result[f"{metric}_acceptable"] = (
            value >= threshold if pd.notna(value) else None
        )

    combined_results.append(result)

combined_df = pd.DataFrame(combined_results)
output_file = "/content/ragas_evaluation_results.csv"
combined_df.to_csv(output_file, index=False, encoding="utf-8-sig")

files.download(output_file)
print("\n✅ 결과 저장 및 다운로드 완료!")


/tmp/ipython-input-3324674693.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipython-input-3324674693.py:3: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
/tmp/ipython-input-3324674693.py:3: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
/tmp/ipython-input-3324674693.py:3: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0.

✅ Metrics 초기화 완료!
✅ Testset 로드 완료: 22개
📊 최종 평가 샘플 수: 22


Evaluating:   0%|          | 0/88 [00:00<?, ?it/s]


📈 메트릭별 평균 점수 (Acceptable 기준 적용)

faithfulness: 0.6073 → ACCEPTABLE ✅ (기준 0.6)
answer_relevancy: 0.2346 → NOT ACCEPTABLE ❌ (기준 0.4)
context_precision: 0.7803 → NOT ACCEPTABLE ❌ (기준 0.8)
context_recall: 0.6815 → NOT ACCEPTABLE ❌ (기준 0.8)

📋 개별 평가 결과 (처음 5개)

[항목 1]
질문: 저는 프리랜서 디자이너로 막 일을 시작했고 주소지는 전남 보성군인데요, 고정 수입이 없어서 취업 상담이나 창업 관련 도움을 받을 곳을 찾고 있어...
답변: 네, 보성읍 새싹길 23에 위치한 보성군 청년센터에서 청년 지원, 취업 상담 및 창업 지원 서비스를 받을 수 있습니다. 프리랜서 디자이너로서 취...
  faithfulness: 1.000 (OK)
  answer_relevancy: 0.489 (OK)
  context_precision: 1.000 (OK)
  context_recall: 1.000 (OK)

[항목 2]
질문: 저 25살인데요 집에서 봉제 납품 일을하는데, 이거 재택으로 일하는 사람에 해당되나요?...
답변: 네, 집에서 봉제 납품 일을 하신다면 재택근로자에 해당됩니다. 재택근로자는 사전에 정해진 보수 산정 방식, 근로시간 등에 따라 자기 집이나 자신...
  faithfulness: 1.000 (OK)
  answer_relevancy: 0.342 (LOW)
  context_precision: 1.000 (OK)
  context_recall: 1.000 (OK)

[항목 3]
질문: 저는 영주시에 살고 입사 1년 차 사회초년생인데요, 요즘 이직이랑 커리어 방향 때문에 고민이 많아서 취업상담 같은거 받아보고 싶거든요. 영주에 ...
답변: 죄송하지만 제공된 컨텍스트에는 영주시의 청년 관련 센터나 취업 상담 지원에 대한 정보가 포함되어 있지 않습니다. 영주시의 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ 결과 저장 및 다운로드 완료!
